# A4. Multi-device networks and batching


Dendra networks can contain populations on different devices. Event and continuous connections are normally placed on the postsynaptic / synapse device, because delivery updates target synaptic state.

## Inspect placement

```python
net.devices()   # population name -> device
net.dtypes()    # population name -> dtype
```

A legacy `net.device()` call returns the first population's device and should not be used as a complete description of heterogeneous networks.

## Mixed CPU / accelerator placement

Within each timestep phase, Dendra queues accelerator-resident components before CPU components. This helps mixed CPU/GPU/MPS networks overlap accelerator kernels with host-side work when the backend permits it.

The dependency phases are still ordered:

```text
NetStim -> continuous target reset -> continuous NetCons -> event NetCons -> populations
```

Only independent components within a phase are reordered by device class.

Cross-device NetCons require moving presynaptic event/gate data to the postsynaptic/synapse device. This can introduce synchronization and copy overhead. Keep tightly coupled populations colocated unless there is a strong reason to split them.

## Batching networks

`Network.batch(n)` creates `n` independent replicas along a leading batch dimension:

```python
net = build_network()
net = net.batch(64)
net.initialize(dt)
net.run(tstop)
```

This is useful for parameter sweeps, Monte Carlo draws, and independent stimulus conditions. It is usually more efficient than building a Python list of independent networks.

## Stimulus shape contract

Let `N` be population size, `C` the compartment count, `T` the number of evaluated timesteps, and `*B` the explicit batch dimensions. A regular Population has shape `[N, C]`; after one or more `batch()` calls it has shape `[*B, N, C]`. The rules below apply equally to a standalone Population and to each named Population in a Network.

Waveforms always return **time last**: a scalar waveform returns `[T]`, while a waveform with value or sweep axes `[*V]` returns `[*V, T]`. Dendra removes that last axis one timestep at a time. The exception is direct precomputed `Population.run(ve=...)`, where time is first: `[T, *V]`.

### Intracellular injections

For `slice.inject(waveform)`, each per-timestep waveform sample must broadcast to the selected shape `S`. Scalars and exact shapes work. Dendra first uses ordinary trailing PyTorch broadcasting. Only if that fails may low-rank data (no more value axes than explicit batch axes) broadcast to the batch shape, right-aligned within those axes, and then be padded with singleton spatial axes. This fallback is enabled only when the complete batch grid remains a leading Cartesian prefix; integer, `None`, boolean, or advanced indexing can remove or reorder axes, so use an explicit trailing-broadcastable shape for those selections.

For an unbatched selection `S=[Ns, Cs]`, common waveform outputs are:

- `[T]`: one waveform shared by the selection;
- `[Cs, T]`: compartment-aligned and shared over selected cells;
- `[Ns, 1, T]`: one waveform per selected cell;
- `[Ns, Cs, T]`: fully specified.

For a batched selection `S=[*B, Ns, Cs]`, the unbatched forms keep their ordinary trailing meaning and are shared across replicas. A low-rank `[*V,T]` waveform may use the batch-only fallback when trailing broadcasting fails. `[*B,1,1,T]` states per-batch, spatially shared intent explicitly, and `[*B,Ns,Cs,T]` is fully specified. Injection before and after `Population.batch()` or `Network.batch()` are both supported; batching rebases existing injection indices.

A frequency sweep is therefore a one-dimensional waveform value axis:

```python
import torch
import dendra as dn
from dendra.units import Hz, nA, ms

B = 9
net = build_windup_network(celsius=36.0, dt_ms=0.0125).batch(B)
frequencies = torch.linspace(10, 90, B) * Hz
stim = dn.mono_rect(amp=1.0 * nA, pw=1 * ms).repeat(frequencies)
net.SG.slice("soma", loc=0.5).inject(stim)
```

Here `stim(t)` has shape `[B,T]`, while the selected soma has spatial shape `[1,1]`. At each timestep `[B]` cannot broadcast to those trailing spatial axes, so the batch fallback expands it to `[B,1,1]` and gives each network replica its own frequency.

### Extracellular inputs

A standalone Population accepts `extra=(ve_s, time)`, where `time` is a Waveform or a time-last tensor. A Network accepts `extra={population_name: (ve_s, waveform)}`; its temporal entry is a Waveform. The spatial and temporal parts broadcast independently:

| Input | Regular Population | After `batch()` | Normalized shape |
| --- | --- | --- | --- |
| spatial `ve_s` | `[C]`, `[1,C]`, `[N,C]` | regular forms stay shared; low-rank `[*V]` may fall back to batch; `[*B,1,1]` is explicit | `[*B,N,C]` |
| temporal Waveform/tensor | `[T]`, `[1,T]`, `[N,T]` | regular forms stay shared; low-rank `[*V,T]` may fall back to batch; `[*B,1,T]` is explicit | `[*B,N,T]` |

For example, the spatial field can be shared while frequency varies by replica:

```python
field = torch.ones(net.SG.np, net.SG.nc)
extra_stim = dn.mono_rect(amp=1.0, pw=1 * ms).repeat(frequencies[:, None])
net.run(1000 * ms, extra={"SG": (field, extra_stim)})  # [B,1,T]
```

Direct precomputed extracellular voltage is a Population-only interface. `Population.step(ve=...)` takes one value normalized to `[*B,N,C]` (a leading singleton timestep is also accepted). `Population.run(ve=...)` takes time first and normalizes the payload after time to `[T,*B,N,C]`: `[T,N,C]` keeps its trailing meaning and is shared across replicas, a low-rank `[T,*V]` may use the batch fallback, `[T,*B,1,1]` is explicitly per-batch, and `[T,*B,N,C]` is fully specified. Networks use the per-population `extra` mapping instead of direct `ve`.

### Ambiguous axes

Ordinary trailing broadcasting wins whenever it works. For a model shaped `[3,N,3]`, a per-step `[3]` is therefore a compartment vector shared over batches; use `[3,1,1]` (or waveform output `[3,1,1,T]`) to state batch intent. If batch size equals `N`, `[N,T]` remains a population-aligned temporal input shared over batches, while `[N,1,T]` is explicitly batch-aligned. With batch shape `[B0,B1]`, fallback data `[B1]` is right-aligned as `[1,B1]` before spatial singleton axes are appended.

## Slot-targeted batching

For banked point-process mechanisms, Dendra keeps two coordinate systems separate during batching:

```text
source_idx: population-flat source coordinate frame
target_idx: synapse-local slot coordinate frame
```

Users should select slots before connecting and let `Network.batch(n)` rebase the source and target coordinate systems.